# Step 3: Deploy AutoGluon TimeSeries Model

Deploys a real-time SageMaker endpoint using the AutoGluon DLC inference container (SDK v3).

The v3 SDK uses a resource-based API: `Model.create()` → `EndpointConfig.create()` → `Endpoint.create()` → `Endpoint.invoke()`.

## Configuration

In [ ]:
import boto3
import sagemaker
import time
import os

from sagemaker.core.helper.session_helper import get_execution_role

REGION = boto3.session.Session().region_name
sess = sagemaker.session.Session()
BUCKET = sess.default_bucket()
S3_PREFIX = "autogluon-timeseries"

AG_VERSION = "1.5"
PY_VERSION = "py312"
INSTANCE_TYPE = "ml.m5.2xlarge"

# After running the pipeline (3-pipeline/pipeline.ipynb), find the training job output:
#   aws s3 ls s3://{BUCKET}/{S3_PREFIX}/pipeline/model/ --recursive
# Update the path below with your pipeline's training output
PIPELINE_MODEL = f"s3://{BUCKET}/{S3_PREFIX}/pipeline/model/<your-pipeline-execution>/output/model.tar.gz"

# Repacked model location (with code/inference.py)
MODEL_DATA = f"s3://{BUCKET}/{S3_PREFIX}/inference/model.tar.gz"

# Inference script to package (serve.py is in the same directory as this notebook)
SERVE_SCRIPT = os.path.abspath("serve.py")

# Resource names
ts = int(time.time())
MODEL_NAME = f"ag-timeseries-{ts}"
ENDPOINT_CONFIG_NAME = f"ag-timeseries-config-{ts}"
ENDPOINT_NAME = f"ag-timeseries-endpoint-{ts}"

# IAM role
ROLE_ARN = get_execution_role()

print(f"Region:         {REGION}")
print(f"Pipeline model: {PIPELINE_MODEL}")
print(f"Repacked model: {MODEL_DATA}")
print(f"Endpoint:       {ENDPOINT_NAME}")

## Resolve Inference Image URI

In [ ]:
from sagemaker.core import image_uris

image_uri = image_uris.retrieve(
    "autogluon",
    region=REGION,
    version=AG_VERSION,
    py_version=PY_VERSION,
    image_scope="inference",
    instance_type=INSTANCE_TYPE,
)
print(f"Inference image: {image_uri}")

## Repackage Model with Inference Code

The AutoGluon DLC uses TorchServe and expects `code/inference.py` inside the model archive.
Pipeline training outputs only contain model artifacts, so we repackage with our `serve.py`.

In [ ]:
import subprocess, tempfile, shutil, tarfile

with tempfile.TemporaryDirectory() as tmpdir:
    local_tar = os.path.join(tmpdir, "model.tar.gz")
    subprocess.run(["aws", "s3", "cp", PIPELINE_MODEL, local_tar], check=True)

    extract_dir = os.path.join(tmpdir, "model")
    os.makedirs(extract_dir)
    with tarfile.open(local_tar) as tf:
        tf.extractall(extract_dir, filter="data")

    code_dir = os.path.join(extract_dir, "code")
    os.makedirs(code_dir, exist_ok=True)
    shutil.copy2(SERVE_SCRIPT, os.path.join(code_dir, "inference.py"))
    print(f"Added code/inference.py from {SERVE_SCRIPT}")

    repack_tar = os.path.join(tmpdir, "model-repack.tar.gz")
    with tarfile.open(repack_tar, "w:gz") as tf:
        for item in os.listdir(extract_dir):
            tf.add(os.path.join(extract_dir, item), arcname=item)

    subprocess.run(["aws", "s3", "cp", repack_tar, MODEL_DATA], check=True)
    print(f"Repacked model uploaded to {MODEL_DATA}")

## Create Model, EndpointConfig, and Endpoint (v3 resource API)

In [ ]:
from sagemaker.core.resources import Model, EndpointConfig, Endpoint
from sagemaker.core.shapes.shapes import ContainerDefinition, ProductionVariant

# 1. Create Model
model = Model.create(
    model_name=MODEL_NAME,
    primary_container=ContainerDefinition(
        image=image_uri,
        model_data_url=MODEL_DATA,
    ),
    execution_role_arn=ROLE_ARN,
)
print(f"Model created: {MODEL_NAME}")

# 2. Create EndpointConfig
endpoint_config = EndpointConfig.create(
    endpoint_config_name=ENDPOINT_CONFIG_NAME,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            model_name=MODEL_NAME,
            initial_instance_count=1,
            instance_type=INSTANCE_TYPE,
        )
    ],
)
print(f"EndpointConfig created: {ENDPOINT_CONFIG_NAME}")

# 3. Create Endpoint
endpoint = Endpoint.create(
    endpoint_name=ENDPOINT_NAME,
    endpoint_config_name=ENDPOINT_CONFIG_NAME,
)
print(f"Endpoint creating: {ENDPOINT_NAME}")

# 4. Wait for InService
endpoint.wait_for_status("InService")
print(f"Endpoint ready: {ENDPOINT_NAME}")

## Send a Sample Prediction

Send a small CSV payload with the last known values for a few items. The model returns forecasts for the configured prediction length.

In [ ]:
import json
import numpy as np
import pandas as pd

# Generate 48 hourly timestamps per item (model needs sufficient history)
items = ["MT_001", "MT_002"]
rows = []
for item in items:
    base = 1500 if item == "MT_001" else 2000
    for h in range(48):
        day = "2014-12-30" if h < 24 else "2014-12-31"
        hour = h % 24
        val = base + np.random.normal(0, 100)
        rows.append({"item_id": item, "timestamp": f"{day} {hour:02d}:00:00", "target": round(val, 1)})

sample_data = pd.DataFrame(rows)
csv_payload = sample_data.to_csv(index=False)
print(f"Sending {len(sample_data)} rows ({len(items)} items x 48 timestamps)")

response = endpoint.invoke(body=csv_payload, content_type="text/csv", accept="application/json")
result = json.loads(response.body.read().decode("utf-8"))
print(f"Forecast keys: {list(result.keys())}")
# Show first 3 mean predictions per item
for item in items:
    preds = {k: v for k, v in result["mean"].items() if item in k}
    print(f"\n{item} forecasts (first 3):")
    for k, v in list(preds.items())[:3]:
        print(f"  {k}: {v:.1f}")

## Cleanup

Uncomment to delete the endpoint when done.

In [ ]:
# endpoint.delete()
# endpoint_config.delete()
# model.delete()
# print("Endpoint, config, and model deleted.")